# 06. 파일럿 데이터셋 생성 (20클립) — **Colab GPU**

샘플링 → 대본 → TTS → 발화 타임스탬프 → 질의/정답 파일까지 **끝까지 한 바퀴**.

- v2: Hard = 판정 생략(K1) + 드문 이벤트 우선 (v1은 태그만으로 Hard → 베이스라인에서 Easy보다 쉬웠음)
- **v3**: 해설위원 추가 (캐스터 2 × 해설 2). Hard에만 방해 코멘트. 질의 시각은 다음 발화 0.1초 전

**실행 전 준비**
- Drive `내 드라이브/kbo-listen/`에 `labels_all.jsonl`(03), `voices/`(참조 음성 4개 + `voices.json`, 05b)
- Colab `/content`에 `kbolisten/` — VS Code 탐색기에서 우클릭 → Upload to Colab (코드라 매번 최신으로)

**결과물 (Drive `kbo-listen/dataset_pilot_v3/`)**
| 파일 | 내용 | 공개 대상 |
|---|---|---|
| `audio/{clip_id}.wav` | 합성 중계 음성 | 참가자 |
| `queries.json` | 클립별 질의 시점 t | 참가자 |
| `gt.json` | 각 t의 정답 B-S-O | 평가 코드 (test는 비공개) |
| `scripts.jsonl` | 발화 텍스트·화자 + 시작/끝 시각 | train만 공개 (test 대본을 주면 ASR 없이 풀림) |
| `meta.jsonl` | split, 트랙, 태그, 목소리, 노브 값 | 분석용 |

In [1]:
# 런타임이 새로 시작됐을 때만
!pip install -q -U qwen-tts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 85.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 111.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
import json
import random
import time
import zlib
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from google.colab import drive
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

from kbolisten.script import KNOBS, queryable, render_clip, tts_text

# 입력과 결과는 Drive에 둔다 (런타임이 바뀌어도 남도록)
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/kbo-listen")
LABELS = DRIVE / "labels_all.jsonl"
VOICE_DIR = DRIVE / "voices"
OUT = DRIVE / "dataset_pilot_v3"
(OUT / "audio").mkdir(parents=True, exist_ok=True)

# 목소리 목록과 참조 문장 (05·05b에서 확정)
VOICES = json.loads((VOICE_DIR / "voices.json").read_text(encoding="utf-8"))
CASTERS = sorted(v for v, info in VOICES.items() if info["role"] == "caster")
ANALYSTS = sorted(v for v, info in VOICES.items() if info["role"] == "analyst")
print("캐스터:", CASTERS, "/ 해설:", ANALYSTS)

with LABELS.open(encoding="utf-8") as f:
    clips = [json.loads(line) for line in f]
print(len(clips), "클립")

Mounted at /content/drive
캐스터: ['caster_m20_v2', 'caster_m30'] / 해설: ['analyst_a40_warm', 'analyst_a45_clear']
46787 클립


In [3]:
# 1) 층화 샘플링
#  Easy: 특수 이벤트 없는 클립
#  Hard: 절반은 드문 이벤트(피치클락·투구 없는 볼넷·주루사) 클립, 나머지는 일반 클립
SPECIAL = {"pitch_clock", "hbp", "ibb", "walk_award", "runner_out"}
RARE = {"pitch_clock", "walk_award", "runner_out"}
MAX_PITCHES = 35  # 너무 긴 반 이닝은 제외
PILOT = {"train": (3, 3), "dev": (2, 2), "test": (5, 5)}  # split: (easy, hard)

# 트랙별 노브: 해설위원은 두 트랙 모두 나오고, 판정 생략과 방해 코멘트는 Hard만
TRACK_KNOBS = {
    "easy": {"p_omit": 0.0, "p_back": 0.10, "p_comment": 0.15, "p_distract": 0.0},
    "hard": {"p_omit": 0.5, "p_back": 0.15, "p_comment": 0.35, "p_distract": 0.5},
}


def sample(clips, plan, seed=42):
    rng = random.Random(seed)
    ok = [c for c in clips if c["n_pitches"] <= MAX_PITCHES]
    picked, used = [], set()

    def take(pool, n, track):
        pool = [c for c in pool if c["clip_id"] not in used]
        chosen = rng.sample(pool, min(n, len(pool)))
        used.update(c["clip_id"] for c in chosen)
        picked.extend({**c, "track": track} for c in chosen)
        return len(chosen)

    for split, (n_easy, n_hard) in plan.items():
        pool = [c for c in ok if c["split"] == split]
        take([c for c in pool if not SPECIAL & set(c["tags"])], n_easy, "easy")
        n_rare = take([c for c in pool if RARE & set(c["tags"])], n_hard // 2, "hard")
        take([c for c in pool if not RARE & set(c["tags"])], n_hard - n_rare, "hard")
    return picked


def assign_voices(clip_id):
    """클립 ID로 캐스터 1명 + 해설 1명을 고정 배정 (4가지 조합이 고르게 나오도록)"""
    h = zlib.crc32(clip_id.encode())
    return {"caster": CASTERS[h % len(CASTERS)], "analyst": ANALYSTS[(h // len(CASTERS)) % len(ANALYSTS)]}


picked = sample(clips, PILOT)

# 2) 트랙별 노브로 대본 생성 + 목소리 배정
for c in picked:
    c["knobs"] = {**KNOBS, **TRACK_KNOBS[c["track"]]}
    c["voices"] = assign_voices(c["clip_id"])
    c["utts"] = render_clip(c, c["knobs"])

for c in picked:
    n = len(c["events"])
    kinds = [u["kind"] for u in c["utts"]]
    print(f"{c['split']:>5} {c['track']:>4} {c['clip_id']}  이벤트 {n:>2}  생략 {n - kinds.count('call'):>2}  "
          f"질의 {len(queryable(c['utts'], n)):>2}  해설 {kinds.count('backchannel') + kinds.count('comment'):>2}  "
          f"방해 {kinds.count('distract'):>2}  {c['voices']['caster']}+{c['voices']['analyst']}  {c['tags']}")

train easy 20240821LTHT02024_08B  이벤트 23  생략  0  질의 23  해설  3  방해  0  caster_m20_v2+analyst_a40_warm  []
train easy 20230530KTHT02023_02T  이벤트 13  생략  0  질의 13  해설  3  방해  0  caster_m20_v2+analyst_a40_warm  []
train easy 20230414LTSS02023_01T  이벤트 11  생략  0  질의 11  해설  2  방해  0  caster_m20_v2+analyst_a45_clear  []
train hard 20240430SKHH02024_07B  이벤트 34  생략 10  질의 28  해설 10  방해  5  caster_m30+analyst_a40_warm  ['walk_award']
train hard 20230806LGSS02023_06B  이벤트 16  생략  4  질의 14  해설  1  방해  2  caster_m30+analyst_a40_warm  []
train hard 20230728LGOB02023_01B  이벤트 12  생략  2  질의 12  해설  7  방해  2  caster_m30+analyst_a45_clear  []
  dev easy 20250429SSSK02025_01T  이벤트 22  생략  0  질의 22  해설  3  방해  0  caster_m30+analyst_a40_warm  []
  dev easy 20250420HTOB02025_04T  이벤트  9  생략  0  질의  9  해설  1  방해  0  caster_m20_v2+analyst_a40_warm  []
  dev hard 20250822WOSS02025_04B  이벤트 21  생략  2  질의 20  해설  7  방해  5  caster_m30+analyst_a45_clear  ['pitch_clock']
  dev hard 20251004SSHT02025_06B  이벤트 11  

In [4]:
# 3) TTS 모델 + 목소리 등록 (목소리마다 자기 참조 문장으로)
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa",
)
prompts = {}
for v, info in VOICES.items():
    ref, ref_sr = sf.read(VOICE_DIR / f"ref_{v}.wav", dtype="float32")
    prompts[v] = model.create_voice_clone_prompt(ref_audio=(ref, ref_sr), ref_text=info["ref_text"])
print("목소리 등록:", list(prompts))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

목소리 등록: ['caster_m30', 'caster_m20_v2', 'analyst_a40_warm', 'analyst_a45_clear']


In [5]:
# 4) 화자별로 한 번씩 배치 합성 → 원래 순서대로 무음을 넣으며 이어 붙이고, 발화마다 시작/끝 시각 기록
LEAD, TAIL = 0.5, 1.0   # 앞뒤 무음(초)
GAP = (0.3, 0.9)        # 발화 사이 무음 범위(초), 클립 ID 시드로 고정


def silence(sec, sr):
    return np.zeros(int(sec * sr), dtype=np.float32)


def synth_clip(c):
    texts = [tts_text(u["text"]) for u in c["utts"]]
    wavs = [None] * len(texts)
    for role in ("caster", "analyst"):
        idx = [i for i, u in enumerate(c["utts"]) if u["speaker"] == role]
        if not idx:
            continue
        out, sr = model.generate_voice_clone(
            text=[texts[i] for i in idx], language=["Korean"] * len(idx),
            voice_clone_prompt=prompts[c["voices"][role]],
        )
        for i, w in zip(idx, out):
            wavs[i] = np.asarray(w, dtype=np.float32)

    rng = random.Random(zlib.crc32(c["clip_id"].encode()))
    pieces = [silence(LEAD, sr)]
    n = len(pieces[0])  # 지금까지 이어 붙인 샘플 수 → 시각은 n / sr로 계산 (누적 오차 없음)
    warnings = []
    for u, text, w in zip(c["utts"], texts, wavs):
        dur = len(w) / sr
        # TTS가 말을 지어내거나(너무 김) 끊어먹는(너무 짧음) 경우 표시
        if dur > 0.35 * len(text) + 1.0 or dur < 0.03 * len(text):
            warnings.append(f"{dur:.1f}초 | {text}")
        u["start"] = round(n / sr, 3)
        pieces.append(w)
        n += len(w)
        u["end"] = round(n / sr, 3)
        gap = silence(rng.uniform(*GAP), sr)
        pieces.append(gap)
        n += len(gap)
    pieces.append(silence(TAIL, sr))
    return np.concatenate(pieces), sr, warnings


total_audio = total_gen = 0.0
for c in picked:
    t0 = time.perf_counter()
    audio, sr, warnings = synth_clip(c)
    gen = time.perf_counter() - t0
    c["duration"] = round(len(audio) / sr, 3)
    sf.write(OUT / "audio" / f"{c['clip_id']}.wav", audio, sr)
    total_audio += c["duration"]
    total_gen += gen
    print(f"{c['clip_id']}: {c['duration']:5.1f}초 음성, {gen:5.1f}초 걸림")
    for w in warnings:
        print("   ⚠", w)

print(f"\n총 음성 {total_audio / 60:.1f}분 / 합성 {total_gen / 60:.1f}분 → RTF {total_gen / total_audio:.2f}")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


20240821LTHT02024_08B: 127.0초 음성,  19.3초 걸림


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


20230530KTHT02023_02T:  74.3초 음성,  18.2초 걸림


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


...

20260408HHSK02026_07T:  92.3초 음성,  20.6초 걸림


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


20260423SKSS02026_01B:  94.9초 음성,  18.4초 걸림


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


20260425LGOB02026_08B:  90.9초 음성,  23.5초 걸림

총 음성 28.1분 / 합성 6.3분 → RTF 0.22


In [6]:
# 5) 질의 시점과 정답
# 이벤트 i의 정보가 다 전달된 뒤(그 이벤트에 딸린 마지막 발화의 끝), 다음 발화가 시작되기 0.1초 전에 묻는다
#  → ASR 단어 시각이 실제보다 늦게 찍혀도 최대한 여유를 준다 (발화 사이 간격은 최소 0.3초)
# 판정 생략 때문에 아직 알 수 없는 이벤트는 묻지 않는다 (queryable)
QUERY_LEAD = 0.1


def make_gt(c):
    last_end = {}
    for u in c["utts"]:
        if u["event_idx"] is not None:
            last_end[u["event_idx"]] = u["end"]  # 판정 → 재언급 → 이닝 종료 순이라 마지막 값이 최대
    starts = [u["start"] for u in c["utts"]]

    gt = []
    for i in queryable(c["utts"], len(c["events"])):
        e, end = c["events"][i], last_end[i]
        nxt = next((s for s in starts if s > end), c["duration"])
        t = max(end + QUERY_LEAD, nxt - QUERY_LEAD)
        gt.append({"t": round(t, 3), "b": e["b"], "s": e["s"], "o": e["o"]})
    return gt


gt = {c["clip_id"]: make_gt(c) for c in picked}
queries = {cid: [q["t"] for q in qs] for cid, qs in gt.items()}

# 질의 시각이 클립 안에서 오름차순인지 확인
for c in picked:
    ts = queries[c["clip_id"]]
    assert ts == sorted(ts) and ts[-1] <= c["duration"], c["clip_id"]
for track in ("easy", "hard"):
    print(track, "질의", sum(len(queries[c["clip_id"]]) for c in picked if c["track"] == track), "개")

easy 질의 158 개
hard 질의 151 개


In [7]:
# 6) 저장
def dump_json(obj, name):
    (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")


dump_json(queries, "queries.json")
dump_json(gt, "gt.json")

with (OUT / "scripts.jsonl").open("w", encoding="utf-8") as f:
    for c in picked:
        f.write(json.dumps({"clip_id": c["clip_id"], "utts": c["utts"]}, ensure_ascii=False) + "\n")

with (OUT / "meta.jsonl").open("w", encoding="utf-8") as f:
    for c in picked:
        meta = {k: c[k] for k in ("clip_id", "split", "track", "tags", "voices", "n_pitches", "duration",
                                  "season", "inning", "half", "away", "home", "knobs")}
        f.write(json.dumps(meta, ensure_ascii=False) + "\n")

print(sorted(p.name for p in OUT.iterdir()))

['audio', 'gt.json', 'meta.jsonl', 'queries.json', 'scripts.jsonl']


In [8]:
# 7) 눈과 귀로 확인: Hard 클립 하나의 대본·시각·질의. 판정이 생략된 이벤트는 (생략)으로 표시
c = next(c for c in picked if c["track"] == "hard" and c["split"] == "test")
print(c["clip_id"], c["tags"], c["voices"])

called = {u["event_idx"] for u in c["utts"] if u["kind"] == "call"}
q_iter = iter(gt[c["clip_id"]])
q = next(q_iter, None)
prev_event = -1
for u in c["utts"]:
    i = u["event_idx"]
    if i is not None:
        # 이 발화 전에 말없이 지나간 투구 (재언급만 있고 판정은 생략된 이벤트 자신도 포함)
        for j in range(prev_event + 1, i + 1):
            if j not in called:
                e = c["events"][j]
                print(f"{'':19}  {'(생략)':>11}  [{e['result']}] → {e['b']}-{e['s']}")
        prev_event = max(prev_event, i)
    who = "해설" if u["speaker"] == "analyst" else "캐스터"
    mark = "  [질의X]" if u.get("query") is False else ""
    print(f"{u['start']:6.2f}~{u['end']:6.2f} {who:<3} {u['kind']:>11}  {u['text']}{mark}")
    while q and u["end"] < q["t"] < u["end"] + 1.0:
        print(f"{'':19}  ▶ 질의 t={q['t']:.2f}  정답 {q['b']}-{q['s']}  {q['o']}아웃")
        q = next(q_iter, None)

display(Audio(str(OUT / "audio" / f"{c['clip_id']}.wav")))

20260331OBSS02026_11B ['pitch_clock', 'walk_award'] {'caster': 'caster_m20_v2', 'analyst': 'analyst_a40_warm'}
  0.50~  6.34 캐스터       intro  11회말, 삼성 공격입니다. 스코어 두산 5 대 삼성 5.
  6.69~  8.45 캐스터     pitcher  마운드에는 양재훈.
  9.31~ 12.19 캐스터      batter  노아웃, 타석에는 최형우.
 13.05~ 15.53 캐스터        call  초구, 바깥쪽 스트라이크.
 15.92~ 17.84 캐스터     restate  카운트 노볼 원 스트라이크.
                     ▶ 질의 t=18.47  정답 0-1  0아웃
 18.57~ 21.13 해설  backchannel  예, 맞습니다.
 21.50~ 23.02 캐스터        call  바깥쪽 스트라이크.
                     ▶ 질의 t=23.57  정답 0-2  0아웃
 23.67~ 26.79 캐스터        call  자, 포크, 쳤습니다! 안타!
                     ▶ 질의 t=27.56  정답 0-0  0아웃
 27.66~ 29.66 해설     distract  볼 배합이 좋네요.
 30.36~ 32.68 캐스터      batter  노아웃, 타석에는 이재현.
 33.33~ 35.33 캐스터        call  초구, 일단 빠집니다.
                     ▶ 질의 t=35.66  정답 1-0  0아웃
 35.76~ 37.12 캐스터        call  번트 파울.
 37.78~ 39.78 캐스터     restate  원볼 원 스트라이크입니다.
                     ▶ 질의 t=40.50  정답 1-1  0아웃
 40.60~ 42.12 해설  backchannel  그렇죠.
 42.87~ 47.02 캐스터        ca